In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/Users/somesh-19583/Desktop/Customer conversion/Source data/prepocessed.csv")
df

,month,day,order,country,page1_main_category,colour,location,model_photography,price,price_2,page,total_clicks,browsing_depth,weekday,weekend
0,6,22,21,29,3,13,1,2,48,1,2,84,4,6,1
1,5,19,6,29,2,13,3,1,57,1,2,9,2,0,0
2,7,15,2,29,3,9,5,1,48,1,1,10,3,1,0
3,5,2,2,29,2,2,4,1,43,2,1,6,2,4,0
4,6,9,16,29,2,9,5,1,57,1,2,15,2,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132374,7,4,3,29,4,2,1,1,48,1,2,5,5,4,0
132375,6,19,9,29,3,14,3,1,28,2,2,33,5,3,0
132376,7,15,4,29,1,3,2,2,38,2,1,8,1,1,0
132377,7,28,16,29,3,9,5,2,20,2,3,18,4,0,0


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# Assume df is your DataFrame already loaded, for example:

# Use 'price' (continuous) as target variable for regression
X = df.drop('price', axis=1)
y = df['price']

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define regression models with pipelines (scaling input features)
models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0, random_state=42))
    ]),
    "Lasso Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=0.1, random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GradientBoostingRegressor(n_estimators=100, random_state=42))
    ])
}

results_test = {}

for name, pipeline in models.items():
    print(f"\n🔹 Training {name}...")

    # Train model
    pipeline.fit(X_train, y_train)

    # Predict on test set
    y_pred_test = pipeline.predict(X_test)

    # Compute regression metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae = mean_absolute_error(y_test, y_pred_test)
    r2 = r2_score(y_test, y_pred_test)

    results_test[name] = {
        "RMSE": rmse,
        "MAE": mae,
        "R-squared": r2
    }

# Convert to DataFrame for neat display
results_df = pd.DataFrame(results_test).T
print("\n📊 Regression Evaluation Results:\n")
print(results_df.round(4))



🔹 Training Linear Regression...

🔹 Training Ridge Regression...

🔹 Training Lasso Regression...

🔹 Training Gradient Boosting...

📊 Regression Evaluation Results:

                     RMSE     MAE  R-squared
Linear Regression  5.8114  4.3676     0.7862
Ridge Regression   5.8114  4.3676     0.7862
Lasso Regression   5.8207  4.3945     0.7856
Gradient Boosting  3.6069  2.4696     0.9177


In [4]:
import pickle

# Extract the decision tree pipeline
gradient_boost = models["Gradient Boosting"]

# Save (pickle) the model to a file
with open("gradient_boostl.pkl", "wb") as file:
    pickle.dump(gradient_boost, file)

print("✅ Gradient Boosting pipeline pickled as 'gradient_boost.pkl'")


✅ Gradient Boosting pipeline pickled as 'gradient_boost.pkl'


In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---------- Load and preprocess test data ----------
test_df = pd.read_csv("/Users/somesh-19583/Desktop/Customer conversion/Source data/test_data.csv")

# Feature engineering
total_clicks_test = test_df.groupby('session_id')['order'].count().reset_index()
total_clicks_test.rename(columns={'order': 'total_clicks'}, inplace=True)

browsing_depth_test = test_df.groupby('session_id')['page'].max().reset_index()
browsing_depth_test.rename(columns={'page': 'browsing_depth'}, inplace=True)

df1_test = total_clicks_test.merge(browsing_depth_test, on='session_id')
test_df = test_df.merge(df1_test, on='session_id')

test_df['weekday'] = pd.to_datetime(test_df[['year', 'month', 'day']]).dt.dayofweek
test_df['weekend'] = (test_df['weekday'] >= 5).astype(int)

test_df = test_df.drop(columns=['year', 'session_id', 'page2_clothing_model'])

# ---------- Prepare features and target ----------
# Make sure your test_df includes the target column 'price_2'
X_test = test_df.drop('price', axis=1)
y_test = test_df['price']

# ---------- Load the pickled Gradient Boosting regression model ----------
with open("/Users/somesh-19583/Desktop/Customer conversion/Pickled Data/gradient_boostl.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# ---------- Predict on the test data ----------
y_pred = pd.DataFrame(loaded_model.predict(X_test))

# ---------- Evaluate regression metrics ----------
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n📌 Sample Predictions on Test Data:", y_pred.head())
print("\n📊 Regression Evaluation Metrics:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R-squared: {r2:.4f}")


FileNotFoundError: [Errno 2] No such file or directory: '/Users/somesh-19583/Desktop/Customer conversion/Source data/test_data.csv.csv'